# Ultra Low RAM Bot.csv Creator
Uses pyarrow **streaming batches** — never loads more than 50k rows at a time.

## CELL 1 — Imports

In [ ]:
import pyarrow.parquet as pq
import pandas as pd
import numpy as np
import gc, os

PARQUET_FILE  = 'NF-BoT-IoT-V2.parquet'
OUTPUT_FILE   = 'bot.csv'
MAX_PER_CLASS = 12_000   # 12k x 5 classes = 60k rows total — very safe for 8GB
BATCH_SIZE    = 50_000   # rows read at a time from parquet

LABEL_MAP = {
    'Benign'         : 'Normal',
    'DDoS'           : 'DDoS',
    'DoS'            : 'DoS',
    'Reconnaissance' : 'Reconnaissance',
    'Theft'          : 'Theft',
}
FINAL_CLASSES = list(LABEL_MAP.values())

print(f'Max rows per class : {MAX_PER_CLASS:,}')
print(f'Max total rows     : ~{MAX_PER_CLASS * len(FINAL_CLASSES):,}')
print('Ready.')

## CELL 2 — Stream parquet in tiny batches → write bot.csv directly

In [ ]:
# Counts how many rows we have per class
class_counts = {cls: 0 for cls in FINAL_CLASSES}
first_write  = True
total_written = 0
batch_num     = 0

# Open parquet as a streaming reader — reads BATCH_SIZE rows at a time
parquet_file = pq.ParquetFile(PARQUET_FILE)

print('Streaming parquet → bot.csv ...')
print(f'Batch size: {BATCH_SIZE:,} rows per batch\n')

# iter_batches streams without loading the full file
for batch in parquet_file.iter_batches(batch_size=BATCH_SIZE):
    batch_num += 1

    # Check if all classes are already full
    if all(class_counts[c] >= MAX_PER_CLASS for c in FINAL_CLASSES):
        print('All classes full — done early!')
        break

    # Convert this tiny batch to pandas
    chunk = batch.to_pandas()

    # Auto-detect label column on first batch
    if batch_num == 1:
        label_col = None
        for c in ['Attack','Label','label','category','Category']:
            if c in chunk.columns:
                label_col = c
                break
        print(f'Label column: "{label_col}"')
        print(f'All columns : {chunk.columns.tolist()}\n')

    # Map labels
    chunk[label_col] = chunk[label_col].map(LABEL_MAP)

    # Keep only rows from our 5 classes
    chunk = chunk[chunk[label_col].isin(FINAL_CLASSES)].copy()

    if chunk.empty:
        del chunk; gc.collect()
        continue

    # For each class, take only as many as still needed
    kept_parts = []
    for cls in FINAL_CLASSES:
        if class_counts[cls] >= MAX_PER_CLASS:
            continue
        sub    = chunk[chunk[label_col] == cls]
        needed = MAX_PER_CLASS - class_counts[cls]
        taken  = sub.head(needed)
        if len(taken) > 0:
            kept_parts.append(taken)
            class_counts[cls] += len(taken)

    del chunk; gc.collect()

    if not kept_parts:
        continue

    save_chunk = pd.concat(kept_parts, ignore_index=True)
    del kept_parts; gc.collect()

    # Rename columns to bot.csv format
    rename_map = {
        'IPV4_SRC_ADDR': 'saddr', 'IPV4_DST_ADDR': 'daddr',
        'L4_SRC_PORT'  : 'sport', 'L4_DST_PORT'  : 'dport',
        label_col      : 'label'
    }
    save_chunk.rename(
        columns={k:v for k,v in rename_map.items() if k in save_chunk.columns},
        inplace=True
    )

    # Drop columns the model never uses
    drop = ['pkSeqID','stime','flgs','attack','state','proto',
            'seq','subcategory','Dataset','FLOW_DURATION_MILLISECONDS']
    save_chunk.drop(columns=[c for c in drop if c in save_chunk.columns], inplace=True)

    # Clean
    save_chunk.replace([np.inf, -np.inf], np.nan, inplace=True)
    save_chunk.fillna(0, inplace=True)
    for col in ['saddr','daddr','sport','dport']:
        if col in save_chunk.columns:
            save_chunk[col] = save_chunk[col].astype(str)

    # Append to CSV (write header only on first write)
    save_chunk.to_csv(
        OUTPUT_FILE,
        mode  = 'w' if first_write else 'a',
        header= first_write,
        index = False
    )
    first_write   = False
    total_written += len(save_chunk)

    del save_chunk; gc.collect()

    # Progress
    status = ' | '.join(f'{c[:5]}:{class_counts[c]:,}' for c in FINAL_CLASSES)
    print(f'Batch {batch_num:03d} | {status} | Total: {total_written:,}')

print(f'\n=== DONE ===')
print(f'Total rows written: {total_written:,}')
print('Final class counts:')
for cls, cnt in class_counts.items():
    print(f'  {cls:<20}: {cnt:,}')

## CELL 3 — Verify bot.csv

In [ ]:
df = pd.read_csv(OUTPUT_FILE)
size_mb = os.path.getsize(OUTPUT_FILE) / 1024 / 1024

print(f'Shape   : {df.shape}')
print(f'Size    : {size_mb:.1f} MB')
print(f'Columns : {df.columns.tolist()}')
print(f'\nClass distribution:')
print(df['label'].value_counts())
print(f'\nSample:')
print(df[['saddr','daddr','sport','dport','label']].head(5).to_string())
print('\n✓ bot.csv ready — plug into your E-GraphSAGE notebook!')